In [3]:
#import required libs
import numpy as np
import pandas as pd
import os
from dotenv import load_dotenv
import json
from mistralai import Mistral

In [4]:
# Load Json Dataset from .json

with open('documents.json', 'rt', encoding='utf-8') as f_out:
    raw_docs = json.load(f_out)

In [8]:
documents = []
for course_dict in raw_docs:
    for doc in course_dict['documents']:
        doc['course'] = course_dict['course']
        documents.append(doc)

In [12]:
from elasticsearch import Elasticsearch

In [14]:
es_client = Elasticsearch('http://localhost:9200')

In [28]:
query = "How can I run Kafka?"

In [29]:
search_query = {
    "size": 5,
    "query": {
        "bool": {
            "must": {
                "multi_match": {
                    "query": query,
                    "fields": ["question^3", "text", "section"],
                    "type": "best_fields"
                }
            },
            "filter": {
                "term": {
                    "course": "data-engineering-zoomcamp"
                }
            }
        }
    }
}

In [20]:
index_name = 'zoom-camp'

In [24]:
es_client.indices.create(index=index_name, body=index_settings)

ObjectApiResponse({'acknowledged': True, 'shards_acknowledged': True, 'index': 'zoom-camp'})

In [21]:
index_settings = {
    "settings": {
        "number_of_shards": 1,
        "number_of_replicas": 0
    },
    "mappings": {
        "properties": {
            "text": {"type": "text"},
            "section": {"type": "text"},
            "question": {"type": "text"},
            "course": {"type": "keyword"} 
        }
    }
}

In [31]:
from tqdm.auto import tqdm

In [33]:
for doc in tqdm(documents):
    es_client.index(index=index_name, document=doc)

  0%|          | 0/948 [00:00<?, ?it/s]

In [34]:
results = es_client.search(index=index_name, body=search_query)

In [37]:
search_results = []

for doc in results['hits']['hits']:
    search_results.append(doc['_source'])

In [43]:
def elastic_search(query):
    search_query = {
        "size": 5,
        "query": {
            "bool": {
                "must": {
                    "multi_match": {
                        "query": query,
                        "fields": ["question^3", "text", "section"],
                        "type": "best_fields"
                    }
                },
                "filter": {
                    "term": {
                        "course": "data-engineering-zoomcamp"
                    }
                }
            }
        }
    }
    answer = []
    results = es_client.search(index=index_name, body=search_query)
    for doc in results['hits']['hits']:
        answer.append(doc['_source'])
    return answer

In [44]:
results_2 = elastic_search(query)

In [54]:
def build_prompt(query, search_results):
    prompt_template = """
You're a course teaching assistant.\n
ANSWER the QUESTION based on the CONTEXT from the FAQ database. \n
Use only the facts from the CONTEXT when answering the QUESTION.\n
If the user's question doesn't contain in the FAQ database, please just kindly decline the requrest and reponse in a kind manner.\n
Don't add any other extra words.\n
QUESTION: {question}

CONTEXT:
{context}
""".strip()
    
    context = ""
    for doc in search_results:
        context = context + f"section: {doc['section']}\nquestion: {doc['question']}\nanswer: {doc['text']}\n\n"
    prompt = prompt_template.format(question=query, context=context)

    return prompt

In [55]:
prompt = build_prompt(query, results_2)

In [56]:
print(prompt)

You're a course teaching assistant.

ANSWER the QUESTION based on the CONTEXT from the FAQ database. 

Use only the facts from the CONTEXT when answering the QUESTION.

If the user's question doesn't contain in the FAQ database, please just kindly decline the requrest and reponse in a kind manner.

Don't add any other extra words.

QUESTION: How can I run Kafka?

CONTEXT:
section: Module 6: streaming with kafka
question: Confluent Kafka: Where can I find schema registry URL?
answer: In Confluent Cloud:
Environment → default (or whatever you named your environment as) → The right navigation bar →  “Stream Governance API” →  The URL under “Endpoint”
And create credentials from Credentials section below it

section: Module 6: streaming with kafka
question: Java Kafka: How to run producer/consumer/kstreams/etc in terminal
answer: In the project directory, run:
java -cp build/libs/<jar_name>-1.0-SNAPSHOT.jar:out src/main/java/org/example/JsonProducer.java

section: Workshop 1 - dlthub
quest

In [68]:
# Connecting LLMs APIs
load_dotenv()

api_key = os.getenv('MISTRAL_API_KEY_2')
client = Mistral(api_key=api_key)

In [70]:
# Buidling chat model
model = "mistral-small-2506"

def llm(prompt):
    try:
        response = client.chat.complete(
            model = model,
            messages = [{
                'role': 'user',
                'content': prompt
            }]
        )
        return response.choices[0].message.content
    except Exception as e:
        print(f"An error occured: {e}")

In [71]:
answer = llm(prompt)

In [72]:
print(answer)

In the project directory, run:
java -cp build/libs/<jar_name>-1.0-SNAPSHOT.jar:out src/main/java/org/example/JsonProducer.java


In [76]:
query = "The course started, Can I still join?"


In [80]:
# Building Rag_bot:
def rag_bot():
    print("--- The Program has started ---")
    print("--- Welcome to the ChatBot ---\n")
    while True:
        query = input("You: ")
        if query != 'exit' and query != 'quit':
            search_res = elastic_search(query)
            prompt = build_prompt(query, search_res)
            answer = llm(prompt)
            print(f"Bot: {answer}\n")
        else:
            print("Bot: GoodBye!")
            print("\n--- End of the Program ---")
            break

In [81]:
rag_bot()

--- The Program has started ---
--- Welcome to the ChatBot ---



You:  What day is today?


Bot: I kindly decline your request as the information is not available in the FAQ database.



You:  What can I learn in DataTalksClub?


Bot: I kindly decline the request.



...

Bot: In the project directory, run:
java -cp build/libs/<jar_name>-1.0-SNAPSHOT.jar:out src/main/java/org/example/JsonProducer.java



You:  What are the coureses?


Bot: I kindly decline your request.



You:  exit


Bot: GoodBye!

--- End of the Program ---


In [82]:
documents[1]

{'text': 'GitHub - DataTalksClub data-engineering-zoomcamp#prerequisites',
 'section': 'General course-related questions',
 'question': 'Course - What are the prerequisites for this course?',
 'course': 'data-engineering-zoomcamp'}